### Validation of Metadata via ITB SHACL API

Sendet die RDF-Dateien direkt an den offiziellen DCAT-AP.de Validator der ITB und parst den zurückgegebenen SHACL-Report.

In [ ]:
from pathlib import Path
import time
import requests
from rdflib import Graph, Namespace, RDF

ITB_API_URL     = "https://www.itb.ec.europa.eu/shacl/dcat-ap.de/api/validate"
VALIDATION_TYPE = "v20_de_spec"
SH = Namespace("http://www.w3.org/ns/shacl#")

DATA_DIR = Path("~/masterthesis/master_thesis_prototype/data/sample_2026-06-18_13-45").expanduser().resolve()
data_files = sorted(DATA_DIR.glob("non_geo_rest.rdf"))

if not data_files:
    raise FileNotFoundError(f"Keine .rdf Dateien in {DATA_DIR}")

print(f"Validiere {len(data_files)} Datei(en) via ITB API...\n")

Validiere 48 Datei(en) via ITB API...



In [ ]:
for data_file in data_files:
    content = data_file.read_text(encoding="utf-8")

    resp = requests.post(
        ITB_API_URL,
        json={
            "contentToValidate": content,
            "embeddingMethod":   "STRING",
            "validationType":    VALIDATION_TYPE,
            "contentSyntax":     "application/rdf+xml",
        },
        headers={"Content-Type": "application/json", "Accept": "text/turtle"},
        timeout=120,
    )
    resp.raise_for_status()

    report = Graph()
    report.parse(data=resp.text, format="turtle")

    conforms = next(
        (bool(o.toPython()) for o in report.objects(None, SH.conforms)),
        True,
    )

    print("=" * 80)
    print(f"File: {data_file.name}  |  Conforms: {conforms}")
    print("=" * 80)

    if not conforms:
        for result in report.subjects(RDF.type, SH.ValidationResult):
            severity  = next(report.objects(result, SH.resultSeverity), None)
            sev_label = str(severity).split("#")[-1] if severity else "?"
            message   = next(report.objects(result, SH.resultMessage), "")
            focus     = next(report.objects(result, SH.focusNode), "")
            path      = next(report.objects(result, SH.resultPath), None)

            print(f"[{sev_label}] {message}")
            print(f"  Focus: {focus}")
            if path:
                print(f"  Path:  {path}")
            print()

    print()
    time.sleep(0.5)  # be polite to the public API

Validierungsbericht für non_geo_rest.rdf:
--------------------------------------------------------------------------------
@prefix dcat: <http://www.w3.org/ns/dcat#> .
@prefix dcatde: <http://dcat-ap.de/def/dcatde/> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

[] a sh:ValidationReport ;
    sh:conforms true ;
    sh:result [ a sh:ValidationResult ;
            sh:focusNode <https://open.rlp.de/dataset/22995426-2d52-4c86-a28f-91b84ebd96fd> ;
            sh:resultMessage "KXX: dcat:Dataset: dcat:landingPage: Zur Steigerung der MetadatenqualitÃ¤t wird die Angabe der ursprÃ¼nglichen Webseite empfohlen. Siehe: https://www.dcat-ap.de/def/dcatde/2.0/spec/#datensatz-ursprungliche-webseite "@de ;
            sh:resultPath dcat:landingPage ;
            sh:resultSeverity sh:Warning ;
            sh:sourceConstraintComponent sh:MinCountConstraintComponent ;
            sh:sourceShape dcatde:KonvXX_dcat_landingPage_w_Min ] .


File: non_geo_rest